# 🎬 Video Studio — من نص إلى فيديو احترافي

**شغّل الخلايا بالترتيب من الأعلى للأسفل.** اضغط ▶ على كل خلية وانتظر علامة ✓.

| الخلية | الوظيفة | الوقت |
|---|---|---|
| 1 | تثبيت الأدوات | ~90 ثانية |
| 2 | فحص سريع (Smoke test) | ~10 ثوانٍ |
| 3 | تحميل محرك الفيديو | فوري |
| 4 | **النص — عدّل هنا** | — |
| 5 | بناء الفيديو | ~20 ثانية لكل مشهد |
| 6 | معاينة وتحميل | — |

> لا يحتاج أي API key. الصوت عبر Google TTS مجاناً.


## 1) التثبيت

In [1]:
#@title تثبيت الأدوات (شغّلها مرة واحدة في كل جلسة)
!apt-get -qq update > /dev/null
!apt-get -qq install -y ffmpeg fonts-noto-core fonts-noto-cjk > /dev/null 2>&1
!pip -q install gTTS arabic-reshaper python-bidi Pillow
!fc-cache -f > /dev/null 2>&1
print("\n✅ تم التثبيت")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.27.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.15 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.

✅ تم التثبيت


## 2) الفحص السريع
هذه الخلية تتأكد أن كل شيء يعمل قبل بناء الفيديو الكامل.

In [2]:
#@title Smoke test — لا تكمل إذا فشلت
import shutil, subprocess, sys, os

checks = []

def check(name, fn):
    try:
        fn(); checks.append((name, True, "")); print(f"✅ {name}")
    except Exception as e:
        checks.append((name, False, str(e))); print(f"❌ {name} → {e}")

check("ffmpeg", lambda: subprocess.run(["ffmpeg","-version"], capture_output=True, check=True))
check("ffprobe", lambda: subprocess.run(["ffprobe","-version"], capture_output=True, check=True))

def _pil():
    from PIL import Image, ImageDraw, ImageFont
    import glob
    fonts = glob.glob("/usr/share/fonts/**/*.ttf", recursive=True)
    assert fonts, "no fonts installed"
    ImageFont.truetype(fonts[0], 40)
check("Pillow + fonts", _pil)

def _tts():
    from gtts import gTTS
    gTTS(text="اختبار", lang="ar").save("/content/_tts_test.mp3")
    assert os.path.getsize("/content/_tts_test.mp3") > 500
check("Google TTS (needs internet)", _tts)

def _render():
    subprocess.run(["ffmpeg","-y","-f","lavfi","-i","color=c=navy:s=320x180:d=2",
                    "-f","lavfi","-i","sine=frequency=300:duration=2",
                    "-c:v","libx264","-c:a","aac","-pix_fmt","yuv420p",
                    "/content/_render_test.mp4","-loglevel","error"], check=True)
    assert os.path.getsize("/content/_render_test.mp4") > 1000
check("H.264 encode", _render)

print()
if all(ok for _, ok, _ in checks):
    print("🎉 كل الفحوصات نجحت — أكمل للخلية التالية")
else:
    print("⚠️ أعد تشغيل الخلية 1، ثم أعد المحاولة")


✅ ffmpeg
✅ ffprobe
✅ Pillow + fonts
✅ Google TTS (needs internet)
✅ H.264 encode

🎉 كل الفحوصات نجحت — أكمل للخلية التالية


## 3) محرك الفيديو
لا تعدّل هذه الخلية إلا إذا أردت تغيير التصميم.

In [3]:
%%writefile /content/video_builder.py
"""
video_builder.py
Automated video pipeline: script -> voiceover -> designed slides -> Ken Burns clips -> final MP4 + SRT.
Designed to run on a fresh Google Colab runtime. No API keys required.
"""

import glob
import json
import os
import re
import subprocess
from pathlib import Path

from PIL import Image, ImageDraw, ImageFilter, ImageFont

BASE = Path("/content")
WORK = BASE / "work"
OUT = BASE / "output"
ASSETS = BASE / "assets"
for _d in (WORK, OUT, ASSETS):
    _d.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

CONFIG = {
    "orientation": "vertical",   # "vertical" (Reels/Shorts/TikTok) or "horizontal" (YouTube)
    "fps": 30,
    "lang": "ar",                # gTTS language code: "ar", "en", "fr", ...
    "brand": "",                 # small footer text, leave "" to hide
    "music": None,               # e.g. "/content/assets/music.mp3"
    "music_volume": 0.10,
    "burn_subtitles": False,     # True = burn the SRT into the video
    "crf": 20,                   # 18 = higher quality/bigger, 23 = smaller
    "transition": 0.45,          # crossfade between scenes (0 = hard cut)
    "scene_gap": 0.30,           # breathing room after each narration line
    "loudness": -16,             # target LUFS (suits social platforms)
    "theme": {
        "bg_top": (14, 20, 38),
        "bg_bottom": (38, 24, 66),
        "accent": (94, 178, 255),
        "title": (255, 255, 255),
        "body": (206, 214, 232),
        "overlay_alpha": 0.55,
        "overlay_color": (10, 12, 24),
    },
}

SIZES = {"vertical": (1080, 1920), "horizontal": (1920, 1080)}


def dims(cfg=CONFIG):
    return SIZES[cfg["orientation"]]


# --------------------------------------------------------------------------
# Text handling (incl. Arabic shaping)
# --------------------------------------------------------------------------

ARABIC = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")


def is_rtl(text: str) -> bool:
    return bool(ARABIC.search(text))


def _has_raqm():
    try:
        from PIL import features
        return features.check("raqm")
    except Exception:
        return False


RAQM = _has_raqm()


def shape(text: str) -> str:
    """
    Prepare Arabic for Pillow.

    When Pillow is built with raqm it already shapes and reorders correctly.
    Pre-reshaping would then emit presentation-form codepoints that many modern
    fonts do not contain, producing empty boxes. So reshape only as a fallback.
    """
    if not is_rtl(text) or RAQM:
        return text
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
        return get_display(arabic_reshaper.reshape(text))
    except Exception:
        return text


def _coverage(path):
    """Codepoints a font can actually draw."""
    try:
        from fontTools.ttLib import TTFont
        f = TTFont(path, fontNumber=0, lazy=True)
        cps = set()
        for t in f["cmap"].tables:
            cps |= set(t.cmap.keys())
        f.close()
        return cps
    except Exception:
        return set()


# Arabic letter + Latin letters + slash + period + digit.
# Arabic-only fonts (Noto Kufi Arabic) fail this and draw Latin as empty boxes.
_REQUIRED = {0x0645, ord("A"), ord("a"), ord("/"), ord("."), ord("0")}
_FONT_CACHE = {}


def _bundled_dejavu(bold=True):
    """
    DejaVu ships inside matplotlib, which is always present in Colab.
    Resolving it through the package avoids depending on system font paths.
    It covers Arabic, Latin, digits and punctuation in one file.
    """
    try:
        import matplotlib
        base = Path(matplotlib.__file__).parent / "mpl-data" / "fonts" / "ttf"
        for nm in (["DejaVuSans-Bold.ttf", "DejaVuSans.ttf"] if bold
                   else ["DejaVuSans.ttf", "DejaVuSans-Bold.ttf"]):
            p = base / nm
            if p.exists():
                return str(p)
    except Exception:
        pass
    return None


def find_font(bold=True) -> str:
    key = bool(bold)
    if key in _FONT_CACHE:
        return _FONT_CACHE[key]

    candidates = []
    p = _bundled_dejavu(bold)
    if p:
        candidates.append(p)

    groups = []
    if bold:
        groups += ["/usr/share/fonts/**/DejaVuSans-Bold.ttf",
                   "/usr/share/fonts/**/*Naskh*Bold*.ttf",
                   "/usr/share/fonts/**/*Amiri*Bold*.ttf",
                   "/usr/share/fonts/**/*Arabic*Bold*.ttf"]
    groups += ["/usr/share/fonts/**/DejaVuSans.ttf",
               "/usr/share/fonts/**/*Naskh*.ttf",
               "/usr/share/fonts/**/*Amiri*.ttf",
               "/usr/share/fonts/**/*Arabic*.ttf",
               "/usr/share/fonts/**/*.ttf"]
    for pat in groups:
        candidates += sorted(glob.glob(pat, recursive=True))

    seen, fallback = set(), None
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        cps = _coverage(c)
        if not cps:
            continue
        if _REQUIRED <= cps:
            _FONT_CACHE[key] = c
            return c
        if fallback is None and 0x0645 in cps:
            fallback = c

    if fallback:
        _FONT_CACHE[key] = fallback
        return fallback
    raise RuntimeError("No usable font found. Run: !apt-get install -y fonts-dejavu-core")


def load_font(size: int, bold=True) -> ImageFont.FreeTypeFont:
    return ImageFont.truetype(find_font(bold), size)


def wrap_lines(draw, text, font, max_width):
    """Wrap on the raw string, shape each finished line afterwards."""
    words = text.split()
    lines, current = [], ""
    for word in words:
        candidate = (current + " " + word).strip()
        if not current or draw.textlength(shape(candidate), font=font) <= max_width:
            current = candidate
        else:
            lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines


# --------------------------------------------------------------------------
# Slide rendering
# --------------------------------------------------------------------------

def gradient(w, h, top, bottom):
    strip = Image.new("RGB", (1, h))
    px = strip.load()
    for y in range(h):
        t = y / max(h - 1, 1)
        px[0, y] = tuple(int(top[i] + (bottom[i] - top[i]) * t) for i in range(3))
    return strip.resize((w, h), Image.BILINEAR)


def _screen(base, layer, amount):
    """Additively blend a soft light layer over the background."""
    from PIL import ImageChops
    lifted = ImageChops.add(base, layer, scale=1.0)
    return Image.blend(base, lifted, amount)


def cover(img, w, h):
    ratio = max(w / img.width, h / img.height)
    img = img.resize((int(img.width * ratio) + 1, int(img.height * ratio) + 1), Image.LANCZOS)
    left = (img.width - w) // 2
    top = (img.height - h) // 2
    return img.crop((left, top, left + w, top + h))


def make_slide(scene, index, total, cfg=CONFIG):
    W, H = dims(cfg)
    th = cfg["theme"]

    bg_path = scene.get("image")
    if bg_path and os.path.exists(bg_path):
        img = cover(Image.open(bg_path).convert("RGB"), W, H)
        overlay = Image.new("RGB", (W, H), tuple(th["overlay_color"]))
        img = Image.blend(img, overlay, th["overlay_alpha"])
    else:
        img = gradient(W, H, tuple(th["bg_top"]), tuple(th["bg_bottom"]))
        img = _screen(img, _blob(W, H, th["accent"]), 0.30)

    draw = ImageDraw.Draw(img)
    margin = int(W * 0.09)
    max_w = W - margin * 2

    title = scene.get("title", "")
    body = scene.get("body", "")
    kicker = scene.get("kicker", "")

    f_kicker = load_font(int(W * 0.032))
    f_title = load_font(int(W * 0.082))
    f_body = load_font(int(W * 0.040), bold=False)

    title_lines = wrap_lines(draw, title, f_title, max_w) if title else []
    body_lines = wrap_lines(draw, body, f_body, max_w) if body else []

    lh_title = int(f_title.size * 1.22)
    lh_body = int(f_body.size * 1.45)
    block_h = len(title_lines) * lh_title + (int(W * 0.05) if body_lines else 0) + len(body_lines) * lh_body
    y = (H - block_h) // 2 + int(H * 0.04)

    rtl = is_rtl(title or body)

    if kicker:
        bar_w, bar_h = int(W * 0.10), max(4, int(W * 0.008))
        bar_x = W - margin - bar_w if rtl else margin
        draw.rounded_rectangle([bar_x, y - int(W * 0.09), bar_x + bar_w, y - int(W * 0.09) + bar_h],
                               radius=bar_h, fill=tuple(th["accent"]))
        kx = W - margin if rtl else margin
        draw.text((kx, y - int(W * 0.072)), shape(kicker), font=f_kicker,
                  fill=tuple(th["accent"]), anchor="ra" if rtl else "la")

    for line in title_lines:
        x = W - margin if rtl else margin
        draw.text((x, y), shape(line), font=f_title, fill=tuple(th["title"]),
                  anchor="ra" if rtl else "la")
        y += lh_title

    if body_lines:
        y += int(W * 0.05)
        for line in body_lines:
            x = W - margin if rtl else margin
            draw.text((x, y), shape(line), font=f_body, fill=tuple(th["body"]),
                      anchor="ra" if rtl else "la")
            y += lh_body

    # progress bar
    pb_y = H - int(H * 0.045)
    draw.rounded_rectangle([margin, pb_y, W - margin, pb_y + max(3, int(W * 0.005))],
                           radius=4, fill=(64, 70, 92))
    filled = margin + int((W - margin * 2) * ((index + 1) / max(total, 1)))
    draw.rounded_rectangle([margin, pb_y, filled, pb_y + max(3, int(W * 0.005))],
                           radius=4, fill=tuple(th["accent"]))

    if cfg.get("brand"):
        f_brand = load_font(int(W * 0.026), bold=False)
        draw.text((W // 2, pb_y - int(W * 0.035)), shape(cfg["brand"]), font=f_brand,
                  fill=(150, 160, 180), anchor="ms")

    path = WORK / f"slide_{index:03d}.png"
    img.save(path, quality=95)
    return str(path)


def _blob(w, h, color):
    layer = Image.new("RGB", (w, h), (0, 0, 0))
    d = ImageDraw.Draw(layer)
    r = int(w * 0.55)
    d.ellipse([-r // 3, -r // 3, r, r], fill=tuple(int(c * 0.45) for c in color))
    d.ellipse([w - r, h - r, w + r // 3, h + r // 3], fill=tuple(int(c * 0.30) for c in color))
    return layer.filter(ImageFilter.GaussianBlur(int(w * 0.16)))


# --------------------------------------------------------------------------
# Audio
# --------------------------------------------------------------------------

def make_voice(text, index, cfg=CONFIG):
    """Synthesise, trim dead air, normalise loudness, add a short tail."""
    from gtts import gTTS
    raw = WORK / f"raw_{index:03d}.mp3"
    path = WORK / f"voice_{index:03d}.m4a"
    gTTS(text=text, lang=cfg["lang"]).save(str(raw))

    gap = cfg.get("scene_gap", 0.30)
    chain = (
        "silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.05"
        ":stop_periods=-1:stop_threshold=-45dB:stop_silence=0.28,"
        f"loudnorm=I={cfg.get('loudness', -16)}:TP=-1.5:LRA=11,"
        f"apad=pad_dur={gap},"
        "afade=t=in:st=0:d=0.12"
    )
    run(["ffmpeg", "-y", "-i", str(raw), "-af", chain,
         "-c:a", "aac", "-b:a", "192k", "-ar", "44100", "-ac", "1", str(path)])
    return str(path)


def media_duration(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=nw=1:nk=1", str(path)],
        capture_output=True, text=True, check=True)
    return float(out.stdout.strip())


# --------------------------------------------------------------------------
# Video assembly
# --------------------------------------------------------------------------

def run(cmd):
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-2500:])
    return proc


def make_clip(slide, voice, index, cfg=CONFIG):
    W, H = dims(cfg)
    fps = cfg["fps"]
    dur = media_duration(voice)
    frames = max(int(dur * fps), 1)
    out = WORK / f"clip_{index:03d}.mp4"

    # alternate pan direction so consecutive scenes don't feel identical
    z = ("min(zoom+0.00042,1.12)" if index % 2 == 0
         else "if(lte(zoom,1.0),1.12,max(1.001,zoom-0.00042))")
    vf = (
        f"scale={int(W*1.3)}:{int(H*1.3)},"
        f"zoompan=z='{z}':d={frames}"
        f":x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={W}x{H}:fps={fps},"
        f"format=yuv420p"
    )
    run([
        "ffmpeg", "-y", "-loop", "1", "-i", slide, "-i", voice,
        "-filter_complex", f"[0:v]{vf}[v]",
        "-map", "[v]", "-map", "1:a",
        "-c:v", "libx264", "-preset", "medium", "-crf", str(cfg["crf"]),
        "-c:a", "aac", "-b:a", "192k", "-ar", "44100",
        "-t", f"{dur:.3f}", str(out),
    ])
    return str(out), dur


def concat(clips, cfg=CONFIG):
    listing = WORK / "clips.txt"
    listing.write_text("".join(f"file '{c}'\n" for c in clips), encoding="utf-8")
    out = WORK / "joined.mp4"
    run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(listing), "-c", "copy", str(out)])
    return str(out), 0.0


def crossfade(clips, durations, cfg=CONFIG):
    """Join clips with a soft dissolve. Returns (path, overlap_seconds)."""
    n = len(clips)
    d = float(cfg.get("transition", 0.45))
    if n < 2 or d <= 0:
        return concat(clips, cfg)

    d = min(d, min(durations) * 0.4)
    cmd = ["ffmpeg", "-y"]
    for c in clips:
        cmd += ["-i", c]

    vparts, aparts = [], []
    vprev, aprev, cum = "[0:v]", "[0:a]", durations[0]
    for i in range(1, n):
        vout = f"[v{i}]" if i < n - 1 else "[vout]"
        aout = f"[a{i}]" if i < n - 1 else "[aout]"
        vparts.append(
            f"{vprev}[{i}:v]xfade=transition=fade:duration={d:.3f}:offset={cum - d:.3f}{vout}")
        aparts.append(f"{aprev}[{i}:a]acrossfade=d={d:.3f}:c1=tri:c2=tri{aout}")
        vprev, aprev = vout, aout
        cum += durations[i] - d

    out = WORK / "joined.mp4"
    cmd += ["-filter_complex", ";".join(vparts + aparts),
            "-map", "[vout]", "-map", "[aout]",
            "-c:v", "libx264", "-preset", "medium", "-crf", str(cfg["crf"]),
            "-c:a", "aac", "-b:a", "192k", "-ar", "44100", str(out)]
    try:
        run(cmd)
        return str(out), d
    except RuntimeError:
        print("  (تعذّر الانتقال الناعم — تم الدمج المباشر)")
        return concat(clips, cfg)


def polish(video, total, cfg=CONFIG):
    """Opening fade, closing fade, stereo audio, web-optimised container."""
    out = WORK / "polished.mp4"
    fo = min(0.9, max(0.3, total * 0.05))
    st = max(total - fo, 0)
    run(["ffmpeg", "-y", "-i", video,
         "-vf", f"fade=t=in:st=0:d=0.5,fade=t=out:st={st:.3f}:d={fo:.3f}",
         "-af", f"afade=t=out:st={st:.3f}:d={fo:.3f}",
         "-c:v", "libx264", "-preset", "medium", "-crf", str(cfg["crf"]),
         "-c:a", "aac", "-b:a", "192k", "-ac", "2", "-ar", "44100",
         "-movflags", "+faststart", str(out)])
    return str(out)


def add_music(video, cfg=CONFIG):
    music = cfg.get("music")
    if not music or not os.path.exists(music):
        return video
    out = WORK / "with_music.mp4"
    run([
        "ffmpeg", "-y", "-i", video, "-stream_loop", "-1", "-i", music,
        "-filter_complex",
        f"[1:a]volume={cfg['music_volume']},afade=t=out:st=0:d=0[m];"
        f"[0:a][m]amix=inputs=2:duration=first:dropout_transition=0[a]",
        "-map", "0:v", "-map", "[a]", "-c:v", "copy",
        "-c:a", "aac", "-b:a", "192k", "-shortest", str(out),
    ])
    return str(out)


def srt_timestamp(seconds):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def write_srt(scenes, durations, path, overlap=0.0, gap=0.30):
    lines, clock = [], 0.0
    for i, (scene, dur) in enumerate(zip(scenes, durations), start=1):
        text = scene.get("narration") or scene.get("title", "")
        end = clock + max(dur - gap, 0.5)
        lines.append(f"{i}\n{srt_timestamp(clock)} --> {srt_timestamp(end)}\n{text}\n")
        clock += dur - overlap
    Path(path).write_text("\n".join(lines), encoding="utf-8")
    return str(path)


def burn_subtitles(video, srt, cfg=CONFIG):
    W, _ = dims(cfg)
    style = (f"FontName=Noto Sans Arabic,FontSize={int(W*0.026)},"
             "PrimaryColour=&H00FFFFFF,OutlineColour=&H90000000,BorderStyle=3,"
             "Outline=2,Shadow=0,MarginV=90,Alignment=2")
    out = WORK / "subtitled.mp4"
    run(["ffmpeg", "-y", "-i", video, "-vf", f"subtitles={srt}:force_style='{style}'",
         "-c:v", "libx264", "-crf", str(cfg["crf"]), "-c:a", "copy", str(out)])
    return str(out)


# --------------------------------------------------------------------------
# Orchestrator
# --------------------------------------------------------------------------

def build(scenes, cfg=CONFIG, name="final"):
    clips, durations = [], []
    n = len(scenes)

    for i, scene in enumerate(scenes):
        narration = scene.get("narration") or scene.get("title", "")
        print(f"[{i+1}/{n}] تصميم الشريحة ...", flush=True)
        slide = make_slide(scene, i, n, cfg)
        print(f"[{i+1}/{n}] التعليق الصوتي ...", flush=True)
        voice = make_voice(narration, i, cfg)
        print(f"[{i+1}/{n}] المعالجة ...", flush=True)
        clip, dur = make_clip(slide, voice, i, cfg)
        clips.append(clip)
        durations.append(dur)

    print("الدمج والانتقالات ...", flush=True)
    video, overlap = crossfade(clips, durations, cfg)
    total = sum(durations) - overlap * (n - 1)

    video = add_music(video, cfg)

    print("اللمسات النهائية ...", flush=True)
    video = polish(video, total, cfg)

    srt = write_srt(scenes, durations, OUT / f"{name}.srt",
                    overlap=overlap, gap=cfg.get("scene_gap", 0.30))
    if cfg.get("burn_subtitles"):
        print("حرق الترجمة ...", flush=True)
        video = burn_subtitles(video, srt, cfg)

    final = OUT / f"{name}.mp4"
    run(["ffmpeg", "-y", "-i", str(video), "-c", "copy",
         "-movflags", "+faststart", str(final)])

    mb = os.path.getsize(final) / 1e6
    print(f"\n✅ تم: {final}\n   المدة {total:.1f} ثانية — الحجم {mb:.1f} MB")
    return str(final), srt


def load_scenes(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


Writing /content/video_builder.py


In [4]:
import importlib, sys
sys.path.insert(0, "/content")
import video_builder as vb
importlib.reload(vb)

f = vb.find_font()
cps = vb._coverage(f)
tests = {"م": "عربي", "A": "لاتيني", "/": "شرطة مائلة", ".": "نقطة",
         "0": "رقم", "،": "فاصلة عربية", "%": "نسبة", "-": "شرطة"}
missing = [n for c, n in tests.items() if ord(c) not in cps]

print("الخط:", f.split("/")[-1])
print("تشكيل تلقائي (raqm):", "نعم" if vb.RAQM else "لا")
if missing:
    print("⚠️ رموز غير مدعومة:", "، ".join(missing))
    print("   شغّل: !apt-get install -y fonts-dejavu-core  ثم أعد هذه الخلية")
else:
    print("✅ المحرك جاهز — كل الرموز مدعومة")


الخط: DejaVuSans-Bold.ttf
تشكيل تلقائي (raqm): نعم
✅ المحرك جاهز — كل الرموز مدعومة


## 4) النص — عدّل هنا فقط

كل عنصر في القائمة = مشهد واحد.

| الحقل | الوظيفة |
|---|---|
| `kicker` | سطر صغير فوق العنوان |
| `title` | العنوان الكبير |
| `body` | نص توضيحي أصغر |
| `narration` | ما يقوله الصوت (إن تُرك فارغاً يُقرأ العنوان) |
| `image` | اختياري: مسار صورة خلفية في `/content/assets/` |


In [5]:
scenes = [
    {
        "kicker": "01 / المقدمة",
        "title": "كيف تصنع فيديو احترافي من الصفر",
        "body": "بدون كاميرا، وبدون برامج مونتاج.",
        "narration": "في هذا الفيديو، سنتعلم كيف نصنع محتوى احترافي بالكامل من نص مكتوب فقط.",
    },
    {
        "kicker": "02 / الفكرة",
        "title": "النص هو الأساس",
        "body": "كل مشهد يتحول تلقائياً إلى شريحة مصممة مع تعليق صوتي.",
        "narration": "الفكرة بسيطة. تكتب النص، والنظام يحوله إلى شرائح مصممة مع تعليق صوتي متزامن.",
    },
    {
        "kicker": "03 / الإخراج",
        "title": "ملف جاهز للنشر",
        "body": "فيديو عمودي عالي الجودة مع ملف ترجمة SRT.",
        "narration": "النتيجة النهائية فيديو جاهز للنشر مباشرة، مع ملف ترجمة منفصل.",
    },
]

# ---- الإعدادات ----
vb.CONFIG["orientation"] = "vertical"      # "vertical" للريلز | "horizontal" لليوتيوب
vb.CONFIG["lang"] = "ar"                   # "en" للإنجليزية
vb.CONFIG["brand"] = ""                    # مثال: "@alfalasi26481"
vb.CONFIG["burn_subtitles"] = False        # True = حرق الترجمة داخل الفيديو
vb.CONFIG["music"] = None                  # مثال: "/content/assets/music.mp3"
vb.CONFIG["transition"] = 0.45             # مدة الانتقال بين المشاهد (0 = قطع حاد)
vb.CONFIG["scene_gap"] = 0.30              # مساحة تنفّس بعد كل جملة
vb.CONFIG["loudness"] = -16                # مستوى الصوت المعياري للمنصات

print(f"✅ {len(scenes)} مشاهد جاهزة")


✅ 3 مشاهد جاهزة


## 5) بناء الفيديو

In [ ]:
final, srt = vb.build(scenes, name="final")


[1/3] تصميم الشريحة ...
[1/3] التعليق الصوتي ...
[1/3] المعالجة ...


## 6) المعاينة والتحميل

In [ ]:
from IPython.display import HTML
import base64, os

size_mb = os.path.getsize(final) / 1e6
print(f"{final}  —  {size_mb:.1f} MB")

if size_mb < 40:
    b64 = base64.b64encode(open(final, "rb").read()).decode()
    display(HTML(f'<video width="320" controls src="data:video/mp4;base64,{b64}"></video>'))
else:
    print("الملف كبير — استخدم زر التحميل أدناه للمعاينة على جهازك")


In [ ]:
from google.colab import files
files.download(final)
files.download(srt)


---
## حل المشاكل

| الخطأ | السبب | الحل |
|---|---|---|
| `ModuleNotFoundError` | انتهت جلسة Colab | أعد تشغيل الخلايا 1 و 3 |
| مربعات ▯▯▯ بدل العربي | الخط ناقص | `!apt-get install -y fonts-noto-core` ثم أعد الخلية 3 |
| `gTTS` يفشل | انقطاع إنترنت أو حد يومي | انتظر دقيقة وأعد المحاولة |
| الصوت مقطوع | نص طويل جداً في مشهد | قسّم `narration` على مشهدين |
| `Session crashed` | ذاكرة ممتلئة | Runtime → Restart، وقلّل عدد المشاهد |

**ملاحظة:** Colab يغلق الجلسة عند الخمول. للأتمتة المجدولة استخدم GitHub Actions بعد أن يعمل هذا يدوياً.
